# Task 2 — Auto Loader: landing -> bronze (Delta)
Read JSON files from landing and loads them to Delta table in `gabrielajaniszews786_bronze`.
Idempotent: the checkpoint tracks which files were already loaded, so re-runs don't duplicate data.
Adds metadata columns (source filename, ingestion timestamp, load date). Trigger `availableNow` processes all available files, then stops (it doesn't run 24/7).

In [0]:
from pyspark.sql.functions import current_timestamp, current_date

CATALOG    = "dbr_dev"
STORAGE_ACCOUNT = "dlspl21databricks"
CONTAINER  = "gabrielajaniszews786"
SCHEMA     = "gabrielajaniszews786_bronze"
BASE       = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"
LANDING    = f"/Volumes/{CATALOG}/{SCHEMA}/entsoe_landing"
SCHEMA_LOC = f"{BASE}/_schema/entsoe_prices" # Schema location
CHECKPOINT = f"{BASE}/_checkpoint/entsoe_prices" # Checkpoint
TARGET     = f"{CATALOG}.{SCHEMA}.entsoe_prices"

In [0]:
df = (spark.readStream
      .format("cloudFiles")
      .option("cloudFiles.format", "json")
      .option("cloudFiles.schemaLocation", SCHEMA_LOC)
      .option("cloudFiles.inferColumnTypes", "true")   # To prevent prices from being read as strings
      .option("cloudFiles.maxFilesPerTrigger", "20")   # Process 50 files at a time
      .load(LANDING)
      # --- Metadata columns required in the task ---
      .selectExpr("*",
                  "_metadata.file_name as source_file",         # source filename
                  "_metadata.file_path as source_path")
      .withColumn("ingestion_ts", current_timestamp())          # ingestion timestamp
      .withColumn("load_date",    current_date()))              # load date

In [0]:
query = (df.writeStream
   .format("delta")
   .option("checkpointLocation", CHECKPOINT) # Indempotency: tracks processed files
   .option("mergeSchema","true")  
   .trigger(once=True)                
   .toTable(TARGET))


In [0]:
print("status:", query.status)
print("exception:", query.exception())
print("lastProgress:", query.lastProgress)

In [0]:
# Checking last progress
print(query.lastProgress["numInputRows"]) if query.lastProgress is not None else print('No progress information available')

In [0]:
import pandas as pd

rows = []
for pr in query.recentProgress:
    src     = pr.get("sources", [{}])[0]
    metrics = src.get("metrics", {})
    rows.append({
        "batchId":            pr.get("batchId"),
        "timestamp":          pr.get("timestamp"),
        "numInputRows":       pr.get("numInputRows"),
        "inputRows/s":        round(pr.get("inputRowsPerSecond", 0) or 0, 1),
        "processedRows/s":    round(pr.get("processedRowsPerSecond", 0) or 0, 1),
        "triggerExec_ms":     pr.get("durationMs", {}).get("triggerExecution"),
        "filesOutstanding":   metrics.get("numFilesOutstanding"),
        "bytesOutstanding":   metrics.get("numBytesOutstanding"),
    })

display(pd.DataFrame(rows))

In [0]:
# Verification check
cnt = spark.table(TARGET).count()
assert cnt > 0, "Bronze empty - ingestion problem"

In [0]:
# Idempotency test

spark.table(TARGET).count()

In [0]:
%sql
SELECT source_file, _rescued_data
FROM dbr_dev.gabrielajaniszews786_bronze.entsoe_prices
WHERE _rescued_data IS NOT NULL

In [0]:
files = dbutils.fs.ls(LANDING)
print("plików w landing:", len(files))
for f in sorted(files, key=lambda x: x.modificationTime)[-8:]:
    print(f.name, f.modificationTime)

In [0]:
spark.sql(f"""
  SELECT bidding_zone, count(*) AS n
  FROM {TARGET}
  GROUP BY bidding_zone
  ORDER BY bidding_zone
""").display()

In [0]:
# Checking active streams

for q in spark.streams.active:
    print(q.name, q.id, q.status)

In [0]:
# Stopping any active streams

for q in spark.streams.active:
    q.stop()